<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Two signals the rule leans on, checked before I trust it:

A: CTR falls as position gets worse. This is the assumption behind FlyRank's CTR-fix logic.

B: CTR is only readable with enough impressions. This decides my 1,000 floor.

In [ ]:
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

DATA = "data/raw/content_refresh_anonymized.csv"
if not Path(DATA).exists():                       # in Colab: fetch the repo, the data ships inside it
    if not Path("flyrank-ml-internship").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Eng7ouda06/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")

df = pd.read_csv(DATA)
FLOOR, MAX_POS, RATIO = 1000, 20, 0.5             # my rule's three settings
BANDS = ["top_3", "page_1", "striking", "page_3_5", "deep"]

def band_of(pos):                                 # avg_position == 0 means "no data", so it gets no band
    return pd.cut(pos.where(pos > 0), [0, 3, 10, 20, 50, np.inf], right=False, labels=BANDS)

df["ctr"] = df.clicks_90d / df.impressions_90d * 100      # CTR in %, rebuilt from raw counts
df["band"] = band_of(df.avg_position)
print(len(df), "pages |", (df.avg_position == 0).sum(), "have no position data")

30000 pages | 1205 have no position data


Verdict A: MIXED. Position matters coarsely: CTR is 0.487% in the top-3 band and 0.036% in the deep band. But it is not clean band by band: page 1 (0.351%) and page 2 (0.354%) are tied, and the top-3 median (0.226%) is below page 1's (0.245%). So I only adjust for position coarsely and do not treat page 1 vs page 2 as different.

In [ ]:
seen = df[df.impressions_90d >= FLOOR]
A = seen.groupby("band", observed=True).agg(n=("content_id", "size"), impressions=("impressions_90d", "sum"),
                                            clicks=("clicks_90d", "sum"), median_ctr=("ctr", "median"))
A["pooled_ctr"] = A.clicks / A.impressions * 100          # total clicks / total impressions
print(A[["n", "pooled_ctr", "median_ctr"]])
print("pooled CTR falls at every band:", A.pooled_ctr.is_monotonic_decreasing)
print("median page CTR falls at every band:", A.median_ctr.is_monotonic_decreasing)

if A.pooled_ctr.is_monotonic_decreasing and A.median_ctr.is_monotonic_decreasing:
    VERDICT_A = "CONFIRMED"
elif A.pooled_ctr.iloc[0] > 2 * A.pooled_ctr.iloc[-1]:
    VERDICT_A = "MIXED"                                    # right direction overall, not band by band
elif A.pooled_ctr.iloc[0] < A.pooled_ctr.iloc[-1]:
    VERDICT_A = "OPPOSITE"
else:
    VERDICT_A = "FALSE"
print("VERDICT A:", VERDICT_A)

             n  pooled_ctr  median_ctr
band                                  
top_3      373    0.486667    0.226180
page_1    6145    0.350864    0.244584
striking  3470    0.354171    0.185782
page_3_5  3334    0.156103    0.095064
deep       190    0.036129    0.000000
pooled CTR falls at every band: False
median page CTR falls at every band: False
VERDICT A: MIXED


Verdict B: CONFIRMED. The share of pages with zero clicks falls at every step: 87% under 50 impressions, 54% at 250–499, 10% at 1,000–4,999, under 1% above 5,000. At a typical CTR of 0.359%, a page with 500 impressions still has a 16.6% chance of zero clicks by luck; at 1,000 it is 2.8%. Below the floor, "low CTR" mostly means "low volume", which is why my floor is 1,000 (not 500).

In [ ]:
low = df[(df.avg_position > 0) & (df.avg_position < MAX_POS)].copy()
low["volume"] = pd.cut(low.impressions_90d, [1, 50, 100, 250, 500, 1000, 5000, np.inf], right=False,
                       labels=["1-49", "50-99", "100-249", "250-499", "500-999", "1,000-4,999", "5,000+"])
B = low.groupby("volume", observed=True).agg(n=("content_id", "size"),
                                             share_with_0_clicks=("clicks_90d", lambda s: (s == 0).mean()))
print(B)

big = low[low.impressions_90d >= FLOOR]
typical = big.clicks_90d.sum() / big.impressions_90d.sum()       # a typical page's CTR (as a fraction)
print(f"typical CTR {typical:.3%}: chance of 0 clicks by pure luck at 500 impressions = {np.exp(-500 * typical):.1%}, at 1,000 = {np.exp(-1000 * typical):.1%}")

VERDICT_B = "CONFIRMED" if B.share_with_0_clicks.is_monotonic_decreasing else "MIXED"
print("VERDICT B:", VERDICT_B)

                n  share_with_0_clicks
volume                                
1-49         4146             0.872648
50-99        1006             0.798211
100-249      1526             0.658585
250-499      1527             0.540275
500-999      2006             0.328016
1,000-4,999  5302             0.098265
5,000+       4686             0.006829
typical CTR 0.359%: chance of 0 clicks by pure luck at 500 impressions = 16.6%, at 1,000 = 2.8%
VERDICT B: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Typical CTR for each band comes from the pages that pass the floor. Every page is ranked; pages where the rule does not fire score 0. The CSV is git-ignored on purpose and this cell regenerates it.

In [ ]:
def score(f):
    """Clicks short of the band's typical CTR. Reads ONLY impressions, clicks and position."""
    band = band_of(f.avg_position).astype(object)
    ctr = f.clicks_90d / f.impressions_90d * 100
    ok = (f.avg_position > 0) & (f.avg_position < MAX_POS) & (f.impressions_90d >= FLOOR)
    tot = f[ok].groupby(band[ok])[["clicks_90d", "impressions_90d"]].sum()
    typical = (tot.clicks_90d / tot.impressions_90d * 100).to_dict()
    expected = band.map(typical)
    fires = ok & (ctr < RATIO * expected)
    return pd.DataFrame({"eligible": ok, "expected_ctr": expected,
                         "score": np.where(fires, f.impressions_90d * (expected - ctr) / 100, 0.0)})

df = df.join(score(df))
df["reason_code"] = np.where(df.score > 0, "low_ctr_for_position", "none")
df["action_label"] = np.where(df.score > 0, "review_title_meta", "monitor")

queue = df.sort_values(["score", "impressions_90d"], ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

os.makedirs("work/outputs", exist_ok=True)
cols = ["rank", "content_id", "client_id", "score", "reason_code", "action_label",
        "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "avg_position", "band"]
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("typical CTR % by band:", df[df.eligible].groupby("band", observed=True).expected_ctr.first().round(3).to_dict())
print(f"{df.eligible.sum():,} eligible pages, {(df.score > 0).sum():,} flagged, {len(queue):,} rows written")
print(queue[cols].head(10))

typical CTR % by band: {'top_3': 0.487, 'page_1': 0.351, 'striking': 0.354}
9,988 eligible pages, 4,068 flagged, 30,000 rows written
   rank            content_id          client_id        score  \
0     1  content_8c19996aa890  client_4e07408562  1693.359395   
1     2  content_8451fc6f034d  client_d029fa3a95  1249.433953   
2     3  content_5fe46e04994d  client_4e07408562  1075.475648   
3     4  content_36ff89c8214e  client_19581e27de   881.389190   
4     5  content_c8e9d6ab9013  client_19581e27de   732.176014   
5     6  content_c84a0ab98e90  client_f369cb89fc   713.377600   
6     7  content_e12868d1f396  client_4e07408562   624.598301   
7     8  content_4a6607efcb46  client_6208ef0f77   606.264182   
8     9  content_cb112fce36be  client_19581e27de   595.362677   
9    10  content_73c54f78c06a  client_f369cb89fc   539.719178   

            reason_code       action_label  impressions_90d  clicks_90d  \
0  low_ctr_for_position  review_title_meta           509252         785   
1

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20: action, reason code, confidence note, and what would make it wrong. Confidence is not a probability: LOW means zero clicks on huge impressions or a client running under 0.6× its expected clicks, MEDIUM means one other doubt applies, HIGH means none of the doubts below apply.

In [ ]:
e = queue[queue.eligible].assign(expected_clicks=lambda x: x.impressions_90d * x.expected_ctr / 100)
client_ratio = e.groupby("client_id").clicks_90d.sum() / e.groupby("client_id").expected_clicks.sum()   # under 1 = client sits below expectation overall

rows = []
for _, r in queue.head(20).iterrows():
    cr = client_ratio[r.client_id]
    doubts = []
    if r.clicks_90d == 0:
        doubts.append(f"zero clicks on {int(r.impressions_90d):,} impressions looks like a tracking or SERP artifact, not a weak title (check Search Console)")
    if cr < 0.75:
        doubts.append(f"its client runs at {cr:.2f}x of expected clicks overall, so the gap may be client-wide")
    if r.days_since_last_update <= 30:
        doubts.append(f"updated {int(r.days_since_last_update)} days ago, so a snippet fix may already be in flight")
    level = "LOW" if (r.clicks_90d == 0 or cr < 0.6) else ("MEDIUM" if doubts else "HIGH")
    rows.append({"rank": r["rank"], "content_id": r.content_id, "client_id": r.client_id, "confidence": level,
                 "note": f"{level}: CTR is {r.ctr / r.expected_ctr:.0%} of typical for {r.band} on {int(r.impressions_90d):,} impressions; its client is at {cr:.2f}x expected clicks",
                 "wrong_if": "; ".join(doubts) or "its queries are mostly branded or answer-box driven, where low CTR is normal"})
review = pd.DataFrame(rows)

for _, r in review.iterrows():
    print(f"#{r['rank']} {r.content_id} | {queue.loc[r['rank'] - 1, 'action_label']} | {queue.loc[r['rank'] - 1, 'reason_code']}")
    print(f"   confidence: {r.note}")
    print(f"   wrong if:   {r.wrong_if}\n")
print(review.confidence.value_counts().to_dict())

#1 content_8c19996aa890 | review_title_meta | low_ctr_for_position
   confidence: MEDIUM: CTR is 32% of typical for top_3 on 509,252 impressions; its client is at 0.90x expected clicks
   wrong if:   updated 20 days ago, so a snippet fix may already be in flight

#2 content_8451fc6f034d | review_title_meta | low_ctr_for_position
   confidence: LOW: CTR is 6% of typical for top_3 on 272,144 impressions; its client is at 0.26x expected clicks
   wrong if:   its client runs at 0.26x of expected clicks overall, so the gap may be client-wide; updated 20 days ago, so a snippet fix may already be in flight

#3 content_5fe46e04994d | review_title_meta | low_ctr_for_position
   confidence: HIGH: CTR is 41% of typical for page_1 on 517,715 impressions; its client is at 0.90x expected clicks
   wrong if:   its queries are mostly branded or answer-box driven, where low CTR is normal

#4 content_36ff89c8214e | review_title_meta | low_ctr_for_position
   confidence: HIGH: CTR is 15% of typical for p

## 4. Weak picks + leakage check

Weak picks: #5 has zero clicks on 208,678 impressions, so check Search Console before rewriting anything. #2 comes from a client running at 0.26× its expected clicks, and #6, #10, #12 and #18 from one client at 0.56×, so those gaps may belong to the clients, not the pages. One client supplies 50% of the top 20 but holds only 38% of eligible pages.

Limits: the "typical CTR" comes from the same snapshot it ranks, so in Week 5 the baseline and the model must be scored against a label from a later window. A queue position means "review this first", not "a rewrite will recover the clicks".

Leakage check: the score reads only impressions, clicks and position (all 90-day totals). It comes out identical with every other column removed, and no trend or last-30/prev-30 column is used.

In [ ]:
print("Weak picks (anything below HIGH confidence):")
print(review[review.confidence != "HIGH"][["rank", "content_id", "confidence", "wrong_if"]].to_string(index=False))

top = queue.head(20).client_id.value_counts()
print(f"\nbiggest client: {top.iloc[0] / 20:.0%} of the top 20 but {(e.client_id == top.index[0]).mean():.0%} of eligible pages")

used = ["impressions_90d", "clicks_90d", "avg_position"]
same = np.allclose(score(df[used]).score, df.score)               # score again with ONLY those 3 columns present
print("score identical when every other column is removed:", same)
banned = ["trend_pct", "trend_direction", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
          "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
print("label-derived or future-window columns in the CSV:", [c for c in banned if c in cols])

Weak picks (anything below HIGH confidence):
 rank           content_id confidence                                                                                                                                           wrong_if
    1 content_8c19996aa890     MEDIUM                                                                                     updated 20 days ago, so a snippet fix may already be in flight
    2 content_8451fc6f034d        LOW its client runs at 0.26x of expected clicks overall, so the gap may be client-wide; updated 20 days ago, so a snippet fix may already be in flight
    5 content_c8e9d6ab9013        LOW                                 zero clicks on 208,678 impressions looks like a tracking or SERP artifact, not a weak title (check Search Console)
    6 content_c84a0ab98e90        LOW its client runs at 0.56x of expected clicks overall, so the gap may be client-wide; updated 20 days ago, so a snippet fix may already be in flight
    7 content_e12868d1f396    